# Day 18 Capstone — Multi-Memory Support Agent
**Agentic Systems Bootcamp · Take-Home**

---

Extends the Day 18 ordering agent into a full **customer support agent**.

| Capstone requirement | Implementation |
|---|---|
| Redis short-term memory | Per-session history, 6-turn window, 1-hr TTL |
| Vector store long-term recall | SQLite + cosine similarity (BOW offline / Voyage-3 live) |
| ≥3 tools + 1 async | `lookup_faq`, `check_inventory`, `create_order`, `approve_order`, `send_confirmation` (async), `check_job`, `verify_order` |
| Routing + chaining | Model picks FAQ vs order flow; chains check → create → approve → verify → confirm |
| Human-approval gate | Orders above $300 require `approve_order()` before confirmation |
| Tracing | `@traced` on every tool; per-turn span table |
| Retries | Exponential backoff (0.1 → 0.2 → 0.4 s) on all tool calls |
| Prompt caching | `cache_control: ephemeral` on system prompt + tools block |

**Extension tasks covered:**
1. Dead-letter queue + retry (Cell 10)
2. `verify_order` evaluator step (Cell 11)
3. Background `threading.Thread` worker (Cell 12)
4. Human-in-the-loop approval gate (Cell 8)
5. Trace spans per call + table (Cell 4)
6. Prompt caching + token savings (Cell 15)

> **Offline mode:** runs fully without an API key using a mock chain.


---
## Cell 1 · Install & configure API key

Create a `.env` file in the same folder as this notebook:
```
ANTHROPIC_API_KEY=sk-ant-...
```
Add `.env` to your `.gitignore` — never commit it.


In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                        'anthropic', 'fakeredis', 'numpy'])

import os
from pathlib import Path

def load_dotenv(path='.env'):
    env_file = Path(path)
    if not env_file.exists():
        return False
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, val = line.split('=', 1)
        os.environ.setdefault(key.strip(), val.strip().strip('"').strip("'"))
    return True

found = load_dotenv()
if not found:
    print('WARNING: .env file not found.')
    print('  Create one with:  ANTHROPIC_API_KEY=sk-ant-...')
elif not os.environ.get('ANTHROPIC_API_KEY'):
    print('WARNING: .env found but ANTHROPIC_API_KEY is missing inside it.')

LIVE  = bool(os.environ.get('ANTHROPIC_API_KEY'))
MODEL = 'claude-sonnet-4-6'
print(f'Mode  : {"LIVE" if LIVE else "OFFLINE mock"}')
print(f'Model : {MODEL}')


Mode  : LIVE
Model : claude-sonnet-4-6


---
## Cell 2 · SQLite database

Three tables: `inventory`, `orders`, `documents` (FAQ embeddings).
The ergonomic chair ($699) is above the $300 approval threshold.


In [2]:
import sqlite3

db = sqlite3.connect(':memory:', check_same_thread=False)
db.executescript("""
    CREATE TABLE inventory (
        sku   TEXT PRIMARY KEY,
        name  TEXT,
        qty   INTEGER,
        price REAL
    );
    CREATE TABLE orders (
        id     INTEGER PRIMARY KEY AUTOINCREMENT,
        sku    TEXT,
        qty    INTEGER,
        total  REAL,
        status TEXT
    );
    CREATE TABLE documents (
        id        INTEGER PRIMARY KEY AUTOINCREMENT,
        content   TEXT,
        metadata  TEXT,
        embedding BLOB
    );
""")

db.executemany('INSERT INTO inventory VALUES (?,?,?,?)', [
    ('KB-01', 'Mechanical keyboard', 12,  129.0),
    ('HUB-2', 'USB-C hub',            0,   58.0),
    ('MON-4', '4K monitor',           5,  410.0),
    ('CHAIR', 'Ergonomic chair',       3,  699.0),
])
db.commit()

print('Tables created. Inventory:')
for row in db.execute('SELECT sku, name, qty, price FROM inventory').fetchall():
    print(f'  {row[0]:<8} {row[1]:<22} qty={row[2]}  ${row[3]}')


Tables created. Inventory:
  KB-01    Mechanical keyboard    qty=12  $129.0
  HUB-2    USB-C hub              qty=0  $58.0
  MON-4    4K monitor             qty=5  $410.0
  CHAIR    Ergonomic chair        qty=3  $699.0


---
## Cell 3 · Redis setup (fakeredis)

Two streams: **`emails`** (main queue) and **`emails:dlq`** (dead-letter queue).


In [3]:
import fakeredis

r = fakeredis.FakeStrictRedis()

try:
    r.xgroup_create('emails', 'mailers', id='0', mkstream=True)
    print('Consumer group "mailers" created on stream "emails"')
except Exception as e:
    print('Group already exists:', e)

print('Redis ready  (streams: emails, emails:dlq)')


Consumer group "mailers" created on stream "emails"
Redis ready  (streams: emails, emails:dlq)


---
## Cell 4 · Tracing  *(Extension Task 5)*

`@traced` records `{tool, args, ms, ok}` for every call.
Call `print_trace_table()` at the end of a turn to see all spans.


In [4]:
import time
import functools

_spans = []

def clear_spans():
    _spans.clear()

def print_trace_table():
    if not _spans:
        print('  (no spans recorded)')
        return
    print(f"  {'Tool':<22} {'Args':<36} {'ms':>6}   OK")
    print('  ' + chr(8212) * 70)
    for s in _spans:
        args_str = str(s['args'])[:34]
        ok_str   = 'OK' if s['ok'] else 'FAIL'
        print(f"  {s['tool']:<22} {args_str:<36} {s['ms']:>6.1f}   {ok_str}")

def traced(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        ok = True
        try:
            result = fn(*args, **kwargs)
            if isinstance(result, dict) and 'error' in result:
                ok = False
            return result
        except Exception:
            ok = False
            raise
        finally:
            _spans.append({
                'tool': fn.__name__,
                'args': kwargs or args[1:],
                'ms':   round((time.perf_counter() - t0) * 1000, 2),
                'ok':   ok,
            })
    return wrapper

print('Tracing ready')


Tracing ready


---
## Cell 5 · Short-term memory (Redis)

Stores the last **6 conversation turn-pairs** per session as JSON in a Redis key.
Each key has a **1-hour TTL** — sessions expire automatically.


In [5]:
import json

MAX_TURNS = 6
TTL_SECS  = 3600

class ShortTermMemory:
    """
    Redis-backed conversation history.
    Key: session:{session_id}:history  ->  JSON list of {role, content} dicts
    """
    def __init__(self, redis_client):
        self._r = redis_client

    def _key(self, session_id):
        return f'session:{session_id}:history'

    def append(self, session_id, role, content):
        key  = self._key(session_id)
        raw  = self._r.get(key)
        hist = json.loads(raw) if raw else []
        hist.append({'role': role, 'content': content})
        if len(hist) > MAX_TURNS * 2:
            hist = hist[-(MAX_TURNS * 2):]
        self._r.set(key, json.dumps(hist), ex=TTL_SECS)

    def get(self, session_id):
        raw = self._r.get(self._key(session_id))
        return json.loads(raw) if raw else []

    def clear(self, session_id):
        self._r.delete(self._key(session_id))


stm = ShortTermMemory(r)

# Smoke-test
stm.append('test', 'user', 'Hello')
stm.append('test', 'assistant', 'Hi!')
print('Short-term memory:', stm.get('test'))
stm.clear('test')


Short-term memory: [{'role': 'user', 'content': 'Hello'}, {'role': 'assistant', 'content': 'Hi!'}]


---
## Cell 6 · Long-term memory (vector store)

Stores FAQ articles as dense vectors in SQLite.
At query time: embed query → cosine similarity against all rows → top-K results.

- **Offline:** 256-dim bag-of-words hash embedding (no API needed)
- **Production:** swap `_embed()` for Voyage-3 via Anthropic


In [6]:
import struct
import numpy as np

def _bow_embed(text, dim=256):
    vec = np.zeros(dim, dtype=np.float32)
    for token in text.lower().split():
        vec[hash(token) % dim] += 1.0
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec

def _cosine(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else 0.0


class LongTermMemory:
    """SQLite vector store with cosine-similarity search."""

    def __init__(self, database, anthropic_client=None):
        self._db     = database
        self._client = anthropic_client

    def _embed(self, text):
        if self._client:
            resp = self._client.embeddings.create(model='voyage-3', input=[text])
            vec  = np.array(resp.embeddings[0].embedding, dtype=np.float32)
            return vec / (np.linalg.norm(vec) or 1)
        return _bow_embed(text)

    def add(self, content, metadata=None):
        vec  = self._embed(content)
        blob = struct.pack(f'{len(vec)}f', *vec.tolist())
        cur  = self._db.execute(
            'INSERT INTO documents (content, metadata, embedding) VALUES (?,?,?)',
            (content, json.dumps(metadata or {}), blob)
        )
        self._db.commit()
        return cur.lastrowid

    def search(self, query, top_k=3):
        q_vec = self._embed(query)
        rows  = self._db.execute(
            'SELECT content, metadata, embedding FROM documents'
        ).fetchall()
        if not rows:
            return []
        scored = []
        for content, meta_json, blob in rows:
            n   = len(blob) // 4
            vec = np.array(struct.unpack(f'{n}f', blob), dtype=np.float32)
            scored.append((_cosine(q_vec, vec), content, json.loads(meta_json)))
        scored.sort(key=lambda x: x[0], reverse=True)
        return scored[:top_k]


ltm = LongTermMemory(db)

FAQ = [
    ('Returns accepted within 30 days for unused items.',            {'cat': 'returns'}),
    ('Shipping takes 3-5 business days. Express available for $15.', {'cat': 'shipping'}),
    ('Orders above $300 require manager approval.',                  {'cat': 'policy'}),
    ('USB-C hub HUB-2 is out of stock. Restock in ~2 weeks.',        {'cat': 'inventory'}),
    ('Bulk orders of 10+ units: contact sales@company.com.',         {'cat': 'bulk'}),
]
for content, meta in FAQ:
    ltm.add(content, meta)

print(f'FAQ seeded with {len(FAQ)} articles')
top = ltm.search('what is the return policy', top_k=1)
print(f'Top result: "{top[0][1]}" (score={top[0][0]:.3f})')


FAQ seeded with 5 articles
Top result: "USB-C hub HUB-2 is out of stock. Restock in ~2 weeks." (score=0.124)


---
## Cell 7 · Sync tools: inventory & orders

### Human-approval gate  *(Extension Task 4)*

```
create_order(total <= $300)  ->  status = 'created'          (stock decremented now)
create_order(total >  $300)  ->  status = 'needs_approval'   (stock held)
                                          |
                                 approve_order(order_id)      (stock decremented here)
```
The agent cannot skip `approve_order` — the system prompt requires it.


In [7]:
APPROVAL_THRESHOLD = 300.0

@traced
def check_inventory(sku):
    row = db.execute(
        'SELECT sku, name, qty, price FROM inventory WHERE sku=? LIMIT 1', (sku,)
    ).fetchone()
    if not row:
        return {'error': f'unknown sku: {sku}'}
    return {'sku': row[0], 'name': row[1], 'qty': row[2], 'price': row[3]}


@traced
def create_order(sku, qty):
    row = db.execute(
        'SELECT qty, price FROM inventory WHERE sku=? LIMIT 1', (sku,)
    ).fetchone()
    if not row:    return {'error': f'unknown sku: {sku}'}
    have, price = row
    if qty <= 0:   return {'error': 'qty must be positive'}
    if have < qty: return {'error': f'insufficient stock: have {have}, need {qty}'}

    total  = round(price * qty, 2)
    status = 'needs_approval' if total > APPROVAL_THRESHOLD else 'created'

    if status == 'created':
        db.execute('UPDATE inventory SET qty=qty-? WHERE sku=?', (qty, sku))

    cur = db.execute(
        'INSERT INTO orders (sku, qty, total, status) VALUES (?,?,?,?)',
        (sku, qty, total, status)
    )
    db.commit()

    result = {'order_id': cur.lastrowid, 'sku': sku, 'qty': qty,
              'total': total, 'status': status}
    if status == 'needs_approval':
        result['next_step'] = f'Call approve_order(order_id={cur.lastrowid}).'
    return result


@traced
def approve_order(order_id):
    row = db.execute(
        'SELECT sku, qty, total, status FROM orders WHERE id=? LIMIT 1', (order_id,)
    ).fetchone()
    if not row: return {'error': f'order {order_id} not found'}
    sku, qty, total, status = row

    if status == 'created':
        return {'order_id': order_id, 'status': 'already_approved', 'total': total}
    if status != 'needs_approval':
        return {'error': f"cannot approve — status is '{status}'"}

    db.execute('UPDATE inventory SET qty=qty-? WHERE sku=?', (qty, sku))
    db.execute("UPDATE orders SET status='created' WHERE id=?", (order_id,))
    db.commit()
    return {'order_id': order_id, 'status': 'approved', 'total': total}


# Quick tests
print('check_inventory KB-01 :', check_inventory('KB-01'))
print('create_order CHAIR x1 :', create_order('CHAIR', 1))   # needs_approval
print('create_order HUB-2 x1 :', create_order('HUB-2', 1))   # out of stock


check_inventory KB-01 : {'sku': 'KB-01', 'name': 'Mechanical keyboard', 'qty': 12, 'price': 129.0}
create_order CHAIR x1 : {'order_id': 1, 'sku': 'CHAIR', 'qty': 1, 'total': 699.0, 'status': 'needs_approval', 'next_step': 'Call approve_order(order_id=1).'}
create_order HUB-2 x1 : {'error': 'insufficient stock: have 0, need 1'}


---
## Cell 8 · Async tool: email confirmation

`send_confirmation` enqueues a job with `XADD` and returns `job_id` in < 1 ms.
The worker (Cell 10 / 12) processes it independently.


In [8]:
import uuid

@traced
def send_confirmation(to, order_id):
    """Enqueue a confirmation email job. Returns job_id immediately (async)."""
    job_id = uuid.uuid4().hex[:8]
    r.xadd('emails', {
        'job_id':  job_id,
        'to':      to,
        'subject': f'Order #{order_id} confirmed',
        'body':    f'Your order #{order_id} is on its way!',
    })
    r.set(f'jobresult:{job_id}', 'queued')
    return {'job_id': job_id, 'status': 'queued'}


@traced
def check_job(job_id):
    """Poll the result key written by the worker."""
    val = r.get(f'jobresult:{job_id}')
    return {'job_id': job_id, 'status': val.decode() if val else 'unknown'}


print('Async email tools ready')


Async email tools ready


---
## Cell 9 · FAQ tool (long-term memory lookup)


In [9]:
@traced
def lookup_faq(query, top_k=3):
    """Semantic search over the FAQ / knowledge base."""
    results = ltm.search(query, top_k=top_k)
    if not results:
        return {'results': [], 'message': 'No FAQ entries found.'}
    return {
        'results': [
            {'score': round(score, 3), 'content': content, 'metadata': meta}
            for score, content, meta in results
        ]
    }


# Spot-check
print('FAQ: "shipping time"')
for item in lookup_faq('how long does shipping take', top_k=2)['results']:
    print(f'  [{item["score"]}] {item["content"]}')


FAQ: "shipping time"
  [0.149] Shipping takes 3-5 business days. Express available for $15.
  [0.0] Returns accepted within 30 days for unused items.


---
## Cell 10 · Email worker with retry & DLQ  *(Extension Task 1)*

Each `run_worker()` call:
1. **Re-claims pending messages** (`XPENDING` + `XCLAIM`) — previously failed jobs retried here
2. **Reads new messages** (`XREADGROUP ">"`) — freshly enqueued jobs

After `MAX_RETRIES=3` failures the job moves to `emails:dlq` and is acknowledged.


In [10]:
MAX_RETRIES = 3

def _is_valid_email(addr):
    return '@' in addr and '.' in addr.split('@')[-1]


def _process(msg_id, fields, stats):
    f        = {k.decode(): v.decode() for k, v in fields.items()}
    job_id   = f['job_id']
    to       = f.get('to', '')
    attempts = int(r.incr(f'jobattempts:{job_id}'))

    if not _is_valid_email(to):
        if attempts >= MAX_RETRIES:
            r.xadd('emails:dlq', {**f, 'failure_reason': 'invalid_email',
                                       'attempts': str(attempts)})
            r.set(f'jobresult:{job_id}', 'dlq')
            r.xack('emails', 'mailers', msg_id)
            stats['dlq'] += 1
        else:
            r.set(f'jobresult:{job_id}', f'failed_attempt_{attempts}')
            stats['failed'] += 1
        return

    r.set(f'jobresult:{job_id}', 'sent')
    r.xack('emails', 'mailers', msg_id)
    stats['processed'] += 1


def run_worker(max_msgs=20):
    """Drain the email stream. Returns {processed, failed, dlq} counts."""
    stats = {'processed': 0, 'failed': 0, 'dlq': 0}

    # Pass 1: re-claim and retry pending messages
    for entry in r.xpending_range('emails', 'mailers', min='-', max='+', count=max_msgs):
        claimed = r.xclaim('emails', 'mailers', 'worker-1',
                           min_idle_time=0, message_ids=[entry['message_id']])
        for msg_id, fields in claimed:
            _process(msg_id, fields, stats)

    # Pass 2: read new messages
    resp = r.xreadgroup('mailers', 'worker-1', {'emails': '>'}, count=max_msgs)
    for _stream, msgs in resp or []:
        for msg_id, fields in msgs:
            _process(msg_id, fields, stats)

    return stats


def inspect_dlq():
    """Peek at the dead-letter queue without consuming messages."""
    return [
        {k.decode(): v.decode() for k, v in fields.items()}
        for _, fields in r.xrange('emails:dlq', count=20)
    ]


print(f'Worker ready  (MAX_RETRIES={MAX_RETRIES})')


Worker ready  (MAX_RETRIES=3)


---
## Cell 11 · Evaluator: `verify_order`  *(Extension Task 2)*

After `create_order` or `approve_order`, the agent calls `verify_order` to confirm:
- order status is `created` (not still `needs_approval`)
- total matches what was quoted
- inventory was actually decremented

This is the **evaluator-optimiser pattern**: the agent cannot call `send_confirmation`
until an independent check passes.


In [11]:
@traced
def verify_order(order_id, expected_total=None):
    """
    Evaluator step: re-reads the order row and confirms it is ready to confirm.
    Returns verified=True/False so the agent knows whether to proceed.
    """
    row = db.execute(
        'SELECT sku, qty, total, status FROM orders WHERE id=? LIMIT 1', (order_id,)
    ).fetchone()

    if not row:
        return {'verified': False, 'error': f'order {order_id} not found'}

    sku, qty, total, status = row

    if status != 'created':
        return {'verified': False,
                'error': f"status is '{status}' — expected 'created'"}

    if expected_total is not None and abs(total - expected_total) > 0.01:
        return {'verified': False,
                'error': f'total mismatch: order=${total}, expected=${expected_total}'}

    inv = db.execute('SELECT qty FROM inventory WHERE sku=? LIMIT 1', (sku,)).fetchone()

    return {
        'verified':        True,
        'order_id':        order_id,
        'sku':             sku,
        'qty':             qty,
        'total':           total,
        'status':          status,
        'stock_remaining': inv[0] if inv else 'unknown',
    }


# Smoke-test
test_od = create_order('KB-01', 1)
print('Order  :', test_od)
print('Verify :', verify_order(test_od['order_id'], expected_total=129.0))
print('Bad id :', verify_order(9999))


Order  : {'order_id': 2, 'sku': 'KB-01', 'qty': 1, 'total': 129.0, 'status': 'created'}
Verify : {'verified': True, 'order_id': 2, 'sku': 'KB-01', 'qty': 1, 'total': 129.0, 'status': 'created', 'stock_remaining': 11}
Bad id : {'verified': False, 'error': 'order 9999 not found'}


---
## Cell 12 · Background worker thread  *(Extension Task 3)*

The agent never calls `run_worker()` directly — a daemon thread polls every second.

```
Agent turn              Worker thread
──────────              ─────────────
send_confirmation()     loop every 1s:
  XADD -> job_id          XREADGROUP
                          process
                          XACK / DLQ
```


In [12]:
import threading

_stop_event    = threading.Event()
_worker_thread = None


def _worker_loop(interval=1.0):
    while not _stop_event.is_set():
        stats = run_worker()
        if any(stats.values()):
            print(f'  [worker thread] {stats}')
        _stop_event.wait(timeout=interval)


def start_worker(interval=1.0):
    global _worker_thread
    _stop_event.clear()
    _worker_thread = threading.Thread(target=_worker_loop,
                                      args=(interval,), daemon=True)
    _worker_thread.start()
    print(f'Worker thread started (polling every {interval}s)')


def stop_worker():
    _stop_event.set()
    if _worker_thread:
        _worker_thread.join(timeout=3)
    print('Worker thread stopped')


# Demo
import time as _time
start_worker(interval=0.5)

jb1 = send_confirmation('thread_a@example.com', order_id=200)
jb2 = send_confirmation('thread_b@example.com', order_id=201)
print(f'Enqueued {jb1["job_id"]} and {jb2["job_id"]}')

_time.sleep(1.5)  # give the thread time to process

print(f'Job 1: {check_job(jb1["job_id"])["status"]}')
print(f'Job 2: {check_job(jb2["job_id"])["status"]}')

stop_worker()


Worker thread started (polling every 0.5s)
Enqueued ff070cd5 and f7ed8f1b
  [worker thread] {'processed': 2, 'failed': 0, 'dlq': 0}
Job 1: sent
Job 2: sent
Worker thread stopped


---
## Cell 13 · Tool schemas & retry dispatcher

The last schema has `cache_control: ephemeral` — this marks the end of the cacheable
prefix (system prompt + all schemas). Repeated turns hit the cache and save ~90% of
input tokens on that prefix.


In [13]:
TOOL_SCHEMAS = [
    {
        'name': 'check_inventory',
        'description': 'Check current stock and price for a product SKU.',
        'input_schema': {
            'type': 'object',
            'properties': {'sku': {'type': 'string'}},
            'required': ['sku'],
        },
    },
    {
        'name': 'create_order',
        'description': (
            'Place an order. Returns needs_approval if total > $300 '
            '-- call approve_order then verify_order before send_confirmation. '
            'Returns error if stock is insufficient.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'sku': {'type': 'string'},
                'qty': {'type': 'integer'},
            },
            'required': ['sku', 'qty'],
        },
    },
    {
        'name': 'approve_order',
        'description': 'Approve a high-value order (needs_approval status). Required before verify_order.',
        'input_schema': {
            'type': 'object',
            'properties': {'order_id': {'type': 'integer'}},
            'required': ['order_id'],
        },
    },
    {
        'name': 'verify_order',
        'description': (
            'Evaluator step: re-reads the order and confirms total and status are correct. '
            'Call after create_order or approve_order, before send_confirmation.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'order_id':       {'type': 'integer'},
                'expected_total': {'type': 'number'},
            },
            'required': ['order_id'],
        },
    },
    {
        'name': 'send_confirmation',
        'description': 'Queue a confirmation email (async). Returns job_id immediately.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'to':       {'type': 'string'},
                'order_id': {'type': 'integer'},
            },
            'required': ['to', 'order_id'],
        },
    },
    {
        'name': 'check_job',
        'description': 'Check the delivery status of a queued email job by job_id.',
        'input_schema': {
            'type': 'object',
            'properties': {'job_id': {'type': 'string'}},
            'required': ['job_id'],
        },
    },
    {
        'name': 'lookup_faq',
        'description': 'Search the FAQ / knowledge base. Call first for any policy or product question.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'query': {'type': 'string'},
                'top_k': {'type': 'integer'},
            },
            'required': ['query'],
        },
        'cache_control': {'type': 'ephemeral'},
    },
]

DISPATCH = {
    'check_inventory':   lambda a: check_inventory(**a),
    'create_order':      lambda a: create_order(**a),
    'approve_order':     lambda a: approve_order(**a),
    'verify_order':      lambda a: verify_order(**a),
    'send_confirmation': lambda a: send_confirmation(**a),
    'check_job':         lambda a: check_job(**a),
    'lookup_faq':        lambda a: lookup_faq(**a),
}


def run_tool(name, args):
    """Dispatch with 3-attempt exponential backoff."""
    fn = DISPATCH.get(name)
    if not fn:
        return {'error': f'unknown tool: {name}'}, True
    for attempt in range(3):
        try:
            out    = fn(args)
            is_err = isinstance(out, dict) and 'error' in out
            return out, is_err
        except Exception as exc:
            if attempt < 2:
                time.sleep(0.1 * (2 ** attempt))
            else:
                return {'error': repr(exc)}, True


print(f'{len(TOOL_SCHEMAS)} tool schemas registered')


7 tool schemas registered


---
## Cell 14 · Agent loop  *(Extension Task 6: prompt caching)*

Four additions on top of Day 18 Colab 2:
1. Load Redis history and prepend to `messages`
2. `cache_control: ephemeral` on system prompt + tools
3. `run_tool()` with retry
4. `print_trace_table()` after every turn


In [14]:
SYSTEM_PROMPT = """\
You are a helpful customer support agent for an electronics store.

Rules:
1. For any policy, shipping, or product question -- call lookup_faq first.
2. To place an order: check_inventory -> create_order.
3. If create_order returns needs_approval, call approve_order next.
4. After create_order or approve_order, ALWAYS call verify_order before send_confirmation.
5. Report order total, verification result, and email job status in your final answer.
"""


def agent(user_text, session_id='default', verbose=True):
    clear_spans()
    history  = stm.get(session_id)
    messages = history + [{'role': 'user', 'content': user_text}]
    stm.append(session_id, 'user', user_text)

    if verbose:
        print(f"\n{'=' * 62}")
        print(f"  USER [{session_id}]: {user_text}")
        print(f"{'-' * 62}")

    # ── Offline mock ───────────────────────────────────────────────────────
    if not LIVE:
        if verbose:
            print('  [OFFLINE] mock chain: lookup_faq -> check_inventory ->')
            print('            create_order -> approve_order -> verify_order -> send_confirmation')
        run_tool('lookup_faq',        {'query': 'high value order approval policy'})
        run_tool('check_inventory',   {'sku': 'CHAIR'})
        od, _ = run_tool('create_order',      {'sku': 'CHAIR', 'qty': 1})
        run_tool('approve_order',     {'order_id': od['order_id']})
        run_tool('verify_order',      {'order_id': od['order_id'], 'expected_total': 699.0})
        jb, _ = run_tool('send_confirmation', {'to': 'demo@example.com',
                                               'order_id': od['order_id']})
        run_worker()
        st, _ = run_tool('check_job',         {'job_id': jb['job_id']})
        reply = (f"(mock) Order #{od['order_id']} -- ${od['total']}, "
                 f"verified, approved, email {st['status']}.")
        if verbose:
            print('\n  TRACE:')
            print_trace_table()
            print(f'\n  AGENT: {reply}')
        stm.append(session_id, 'assistant', reply)
        return reply

    # ── Live loop ──────────────────────────────────────────────────────────
    from anthropic import Anthropic
    client = Anthropic()

    for _step in range(12):
        resp = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            system=[{
                'type': 'text',
                'text': SYSTEM_PROMPT,
                'cache_control': {'type': 'ephemeral'},
            }],
            tools=TOOL_SCHEMAS,
            messages=messages,
        )

        if resp.stop_reason == 'tool_use':
            messages.append({
                'role': 'assistant',
                'content': [b.model_dump() for b in resp.content],
            })
            results = []
            for block in resp.content:
                if block.type == 'tool_use':
                    if verbose:
                        print(f'  -> {block.name}({block.input})')
                    out, is_err = run_tool(block.name, block.input)
                    results.append({
                        'type':        'tool_result',
                        'tool_use_id': block.id,
                        'content':     json.dumps(out),
                        'is_error':    is_err,
                    })
            messages.append({'role': 'user', 'content': results})
            run_worker()
            continue

        reply = ''.join(b.text for b in resp.content if b.type == 'text')
        stm.append(session_id, 'assistant', reply)

        if verbose:
            u = getattr(resp, 'usage', None)
            if u:
                read    = getattr(u, 'cache_read_input_tokens', 0)
                written = getattr(u, 'cache_creation_input_tokens', 0)
                print(f'  [cache] read={read}  written={written}')
            print('\n  TRACE:')
            print_trace_table()
            print(f'\n  AGENT: {reply}')

        return reply

    return '(max steps reached)'


print('Agent ready')


Agent ready


---
## Cell 15 · Demo — Turn 1: FAQ question

The agent routes to `lookup_faq` and answers without placing any order.


In [15]:
agent('What is your return policy?', session_id='demo')



  USER [demo]: What is your return policy?
--------------------------------------------------------------
  -> lookup_faq({'query': 'return policy', 'top_k': 3})
  [cache] read=0  written=0

  TRACE:
  Tool                   Args                                     ms   OK
  ——————————————————————————————————————————————————————————————————————
  lookup_faq             {'query': 'return policy', 'top_k'      0.4   OK

  AGENT: Here's our **return policy**:

- 📦 **Returns are accepted within 30 days** of purchase.
- Items must be **unused** to qualify for a return.

If you'd like to initiate a return or have any further questions about the process, feel free to ask! 😊


"Here's our **return policy**:\n\n- 📦 **Returns are accepted within 30 days** of purchase.\n- Items must be **unused** to qualify for a return.\n\nIf you'd like to initiate a return or have any further questions about the process, feel free to ask! 😊"

---
## Cell 16 · Demo — Turn 2: Low-value order ($258, no approval)

Chain: `check_inventory` -> `create_order` -> `verify_order` -> `send_confirmation`


In [16]:
agent('Order 2 keyboards KB-01, email asha@example.com', session_id='demo')



  USER [demo]: Order 2 keyboards KB-01, email asha@example.com
--------------------------------------------------------------
  -> check_inventory({'sku': 'KB-01'})
  -> create_order({'sku': 'KB-01', 'qty': 2})
  -> verify_order({'order_id': 3, 'expected_total': 258.0})
  -> send_confirmation({'to': 'asha@example.com', 'order_id': 3})
  [cache] read=0  written=0

  TRACE:
  Tool                   Args                                     ms   OK
  ——————————————————————————————————————————————————————————————————————
  check_inventory        {'sku': 'KB-01'}                        0.0   OK
  create_order           {'sku': 'KB-01', 'qty': 2}              0.3   OK
  verify_order           {'order_id': 3, 'expected_total':       0.1   OK
  send_confirmation      {'to': 'asha@example.com', 'order_      0.6   OK

  AGENT: Everything is all set! Here's a summary:

- 🛒 **Order ID:** 3
- ⌨️ **Item:** Mechanical Keyboard (KB-01) x2
- 💰 **Order Total:** $258.00
- ✅ **Verification:** Passed
- 📧 *

"Everything is all set! Here's a summary:\n\n- 🛒 **Order ID:** 3\n- ⌨️ **Item:** Mechanical Keyboard (KB-01) x2\n- 💰 **Order Total:** $258.00\n- ✅ **Verification:** Passed\n- 📧 **Confirmation Email:** Queued to asha@example.com (Job ID: `89806ea4`)\n\nLet me know if there's anything else I can help you with!"

---
## Cell 17 · Demo — Turn 3: High-value order ($699, approval gate fires)

Chain: `check_inventory` -> `create_order` -> **`approve_order`** -> `verify_order` -> `send_confirmation`


In [17]:
agent('Order 1 ergonomic chair CHAIR, confirm to priya@example.com', session_id='demo')



  USER [demo]: Order 1 ergonomic chair CHAIR, confirm to priya@example.com
--------------------------------------------------------------
  -> check_inventory({'sku': 'CHAIR'})
  -> create_order({'sku': 'CHAIR', 'qty': 1})
  -> approve_order({'order_id': 4})
  -> verify_order({'order_id': 4, 'expected_total': 699.0})
  -> send_confirmation({'to': 'priya@example.com', 'order_id': 4})
  [cache] read=0  written=0

  TRACE:
  Tool                   Args                                     ms   OK
  ——————————————————————————————————————————————————————————————————————
  check_inventory        {'sku': 'CHAIR'}                        0.3   OK
  create_order           {'sku': 'CHAIR', 'qty': 1}              0.1   OK
  approve_order          {'order_id': 4}                         0.3   OK
  verify_order           {'order_id': 4, 'expected_total':       0.3   OK
  send_confirmation      {'to': 'priya@example.com', 'order      0.6   OK

  AGENT: Everything is all set! Here's a summary:

- 🪑 **

"Everything is all set! Here's a summary:\n\n- 🪑 **Order ID:** 4\n- **Item:** Ergonomic Chair (CHAIR) x1\n- 💰 **Order Total:** $699.00\n- ✅ **Approval:** Granted (high-value order)\n- ✅ **Verification:** Passed\n- 📧 **Confirmation Email:** Queued to priya@example.com (Job ID: `e4b397a2`)\n\nLet me know if there's anything else I can help with!"

---
## Cell 18 · Inspect final state


In [18]:
print('Orders:')
for row in db.execute('SELECT id, sku, qty, total, status FROM orders').fetchall():
    print(f'  #{row[0]}  {row[1]}  qty={row[2]}  ${row[3]}  [{row[4]}]')

print('\nInventory remaining:')
for row in db.execute('SELECT sku, name, qty FROM inventory').fetchall():
    print(f'  {row[0]:<8} {row[1]:<22} qty={row[2]}')

print('\nSession history (demo):')
for m in stm.get('demo'):
    print(f'  [{m["role"]:>9}]  {m["content"][:70]}')

print('\nDead-letter queue:')
dlq = inspect_dlq()
print(f'  {len(dlq)} item(s)' if dlq else '  (empty)')


Orders:
  #1  CHAIR  qty=1  $699.0  [needs_approval]
  #2  KB-01  qty=1  $129.0  [created]
  #3  KB-01  qty=2  $258.0  [created]
  #4  CHAIR  qty=1  $699.0  [created]

Inventory remaining:
  KB-01    Mechanical keyboard    qty=9
  HUB-2    USB-C hub              qty=0
  MON-4    4K monitor             qty=5
  CHAIR    Ergonomic chair        qty=2

Session history (demo):
  [     user]  What is your return policy?
  [assistant]  Here's our **return policy**:

- 📦 **Returns are accepted within 30 da
  [     user]  Order 2 keyboards KB-01, email asha@example.com
  [assistant]  Everything is all set! Here's a summary:

- 🛒 **Order ID:** 3
- ⌨️ **I
  [     user]  Order 1 ergonomic chair CHAIR, confirm to priya@example.com
  [assistant]  Everything is all set! Here's a summary:

- 🪑 **Order ID:** 4
- **Item

Dead-letter queue:
  (empty)


---
## Cell 19 · DLQ demo — inject a bad email address

After 3 worker runs the job should appear in `emails:dlq`.


In [19]:
bad_id = uuid.uuid4().hex[:8]
r.xadd('emails', {'job_id': bad_id, 'to': 'notanemail',
                  'subject': 'Test DLQ', 'body': 'This should fail'})
r.set(f'jobresult:{bad_id}', 'queued')
print(f'Injected job {bad_id}  to=notanemail\n')

for i in range(1, MAX_RETRIES + 1):
    stats  = run_worker()
    status = check_job(bad_id)['status']
    print(f'  Worker run {i}: stats={stats}  job={status}')

print('\nDLQ contents:')
for item in inspect_dlq():
    print(f'  {item}')


Injected job f557520d  to=notanemail

  Worker run 1: stats={'processed': 0, 'failed': 1, 'dlq': 0}  job=failed_attempt_1
  Worker run 2: stats={'processed': 0, 'failed': 1, 'dlq': 0}  job=failed_attempt_2
  Worker run 3: stats={'processed': 0, 'failed': 0, 'dlq': 1}  job=dlq

DLQ contents:
  {'job_id': 'f557520d', 'to': 'notanemail', 'subject': 'Test DLQ', 'body': 'This should fail', 'failure_reason': 'invalid_email', 'attempts': '3'}


---
## Architecture summary

```
User
 |
 v
agent(user_text, session_id)
 |-- stm.get()           load Redis history (short-term memory)
 |-- claude-sonnet-4-6   cached system prompt + tool schemas
 |-- run_tool()          dispatch with 3x retry + backoff
 |    |-- lookup_faq          -> LongTermMemory (cosine search)
 |    |-- check_inventory     -> SQLite read
 |    |-- create_order        -> SQLite write  -+- approval gate
 |    |-- approve_order       -> SQLite write  -+  if total > $300
 |    |-- verify_order        -> SQLite read   (evaluator step)
 |    |-- send_confirmation   -> Redis XADD    (async)
 |    +-- check_job           -> Redis GET
 |-- run_worker()         OR  background thread (Cell 12)
 +-- stm.append()         store reply in Redis
```

| Extension task | Cell | Pattern |
|---|---|---|
| Dead-letter queue + retry | 10, 19 | At-least-once delivery with DLQ |
| `verify_order` evaluator | 11 | Evaluator-optimiser |
| Background worker thread | 12 | True async decoupling |
| Human-approval gate | 7, 17 | Human-in-the-loop |
| Trace spans + table | 4 | Observability |
| Prompt caching | 13, 14 | Token efficiency |
